# EXP-2026-008 / Q5-E — PREP P3: source-match equivalence differential

**이 노트북은 미실행 상태로 커밋된다** — 저장소 사본은 출력이 전부 비어 있고
`execution_count` 가 모두 `null` 이다. 실행 승인이 나기 전까지는 그대로 둔다.

**실행 승인은 아직 없다.** 이 노트북은 P3 *구현* 이 무엇을 할지 보여줄 뿐이고,
등록 `data.py` 를 열지 않는다. 열려면 두 장벽이 **둘 다** 열려야 한다.

| 장벽 | 지금 값 | 여는 방법 |
|---|---|---|
| `P3.OPEN_REGISTERED_DATA` | `False` | 호출 지점에서 명시적으로 opt-in |
| `P3.EXECUTION_APPROVAL_RECORD['granted']` | `False` | 별도 승인 PR 이 한 필드를 바꾼다 |

Q5-E 실행 토큰과 P1/P2 PREP 토큰은 **이름으로 거부** 된다. 다른 단계의 승인이
이 단계를 열 수 없다.

## 무엇을 확인하는가

`Q5E.match_peaks_to_annotations()` 는 산문에서 옮겨 적은 **후보 어댑터** 다.
같은 오독을 두 번 하면 "일치" 가 나오므로, oracle 은 규칙을 다시 옮겨 적은
두 번째 구현이 **아니다** — digest 로 검증한 등록 `data.py` 자체를 합성
의존성 주입 아래에서 실행하고, 그 실행 자취에서 결정을 기계적으로 읽는다.

비교 대상: peak↔annotation 매핑 · kept row 집합과 **순서** · 소비된 주석과
**소비 시점** · 반환(release) 여부 · 미매칭 주석 · 미매칭 peak · AAMI 선택
전/후 · 경계컷 전/후.

**22/22 record count 재현은 증명이 아니다.** 이 노트북은 실제 record count 를
열지 않으며, 여러 어댑터를 돌려 좋은 것을 고르는 기능도 없다.


In [ ]:
# 1. DESIGN — 실행 순서를 먼저 보여준다. 이 셀은 아무것도 열지 않는다.
STAGES = [
    '1. DESIGN                              이 표와 경계 선언',
    '2. ENVIRONMENT                         repo/모듈 로드와 staleness guard',
    '3. SYNTHETIC_FIXTURES                  합성 자산만으로 회귀 스위트 실행',
    '4. DEPENDENCY_AND_APPROVAL_PREFLIGHT   의존성·장벽 점검(자격증명 이전)',
    '5. SOURCE_FILE_IDENTITY                file id → inventory → 바이트 SHA-256',
    '6. SOURCE_ORACLE_DIFFERENTIAL          등록 build_record 실행 ↔ 어댑터',
    '7. RESULT_GATE                         등록 gate 로 후보 구조 검사(등록 아님)',
    '8. BUNDLE_REPORT                       외부 동결용 digest 와 다음 단계',
]
for line in STAGES:
    print(line)
print()
print('승인 전에는 1~4 까지만 진행된다. 5 이후는 두 장벽이 모두 열려야 한다.')
print('이 노트북은 SOURCE_MATCH_ORACLE_RECORD 에 아무것도 쓰지 않는다 —')
print('등록은 Codex 인수 뒤 별도 registration PR 의 몫이다.')


In [ ]:
# 2. ENVIRONMENT — 저장소를 찾아 로드하고, 쓰려는 기능이 실제로 있는지 본다.
#    '경로가 있다' 가 아니라 '필요한 모듈이 거기 있다' 로 판정한다(quest56 과 동일).
import os, sys, json, subprocess

REPO = ''          # 절대경로를 알면 여기에 적는다 (그러면 탐색 생략)
REPO_URL = 'https://github.com/ehdbddl06001-ui/my-github-test'


def _looks_like_repo(path):
    return os.path.isfile(os.path.join(
        path, 'mit-bih', 'q5e_prep_p3_source_match_equivalence.py'))


if not REPO:
    for candidate in ('/content/repo', os.getcwd(),
                      os.path.abspath(os.path.join(os.getcwd(), '..'))):
        if _looks_like_repo(candidate):
            REPO = candidate
            break

if not REPO:
    REPO = '/content/repo'
    if not os.path.isdir(REPO):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO],
                       check=True)
    if not _looks_like_repo(REPO):
        raise RuntimeError('clone 은 됐는데 P3 모듈이 없다 — 브랜치를 확인하라')

sys.path.insert(0, os.path.join(REPO, 'mit-bih'))
import q5d_order_preserving_beat_join as BJ
import q5e_leg2_failure_mechanism_audit as Q5E
import q5e_prep_p1_p2_asset_identity as P12
import q5e_prep_p3_source_match_equivalence as P3

missing = [name for name in P3.module_capabilities() if not hasattr(P3, name)]
if missing:
    raise RuntimeError(f'낡은 사본이다 — 없는 기능: {missing}')

print('repo :', REPO)
print('P3   :', Q5E.sha256_file(P3.__file__))
print('Q5E  :', Q5E.sha256_file(Q5E.__file__))
print('Q5D  :', Q5E.sha256_file(BJ.__file__), '(frozen)')
print()
print(P3.design_card())


In [ ]:
# 3. SYNTHETIC_FIXTURES — 등록 자산을 열기 전에 합성 자산만으로 전 경로를 점검한다.
#    여기서 깨지면 승인을 켜기 전에 고치는 편이 훨씬 싸다. 이 셀은 Drive 를
#    호출하지 않고 등록 data.py 도 열지 않는다 — 합성 producer 로만 돈다.
TEST = os.path.join(REPO, 'mit-bih',
                    'test_q5e_prep_p3_source_match_equivalence.py')
proc = subprocess.run([sys.executable, TEST], capture_output=True, text=True)
print(proc.stdout or '(stdout 없음)')
if proc.returncode != 0:
    print(proc.stderr[-4000:])
    raise RuntimeError(f'합성 회귀 스위트 실패 (exit {proc.returncode})')

print()
print('여섯 required fixture 와 각각이 반증하는 것:')
for name in P3.fixture_names():
    card = P3.fixture_card(name)
    print(' -', name)
    print('     peaks      :', card['peaks'])
    print('     annotations:', card['annotations'])
    print('     반증 대상  :', card['refutes'])
print()
print('required set 일치:',
      list(P3.fixture_names()) == list(Q5E.SOURCE_MATCH_REQUIRED_FIXTURES))
print('fixture 에 정답을 적지 않는다 — 정답은 등록 source 가 말한다.')


In [ ]:
# 4. DEPENDENCY_AND_APPROVAL_PREFLIGHT — **자격증명 생성 이전** 에 점검한다.
#    의존성 확인을 credential 뒤에 두면, 없는 패키지 때문에 실패하는 실행이
#    이미 토큰을 발급받은 상태가 된다. 순서 자체가 계약이다.
report = P12.check_runtime_dependencies()
print('필요 패키지 :', report['required'])
print('없는 것     :', report['missing'])
print('설치는 하지 않는다 —', report['note'])
print()
print('장벽 1  OPEN_REGISTERED_DATA         :', P3.OPEN_REGISTERED_DATA)
print('장벽 2  EXECUTION_APPROVAL_RECORD    :',
      P3.EXECUTION_APPROVAL_RECORD['granted'])
print('요구 scope                           :', P3.DRIVE_READONLY_SCOPE)
print('거부되는 다른 단계 토큰              :', len(P3.REFUSED_TOKENS), '개')
for note in P3.REFUSED_TOKENS.values():
    print('   -', note.split('.')[0])
print()
# 승인 토큰은 비워 둔 채 커밋한다. 승인이 나면 그때 사용자가 채운다.
APPROVAL = None
OPEN_REGISTERED_DATA = False
print('APPROVAL 설정 여부 :', P3.execution_is_approved(APPROVAL))
print(P3.APPROVAL_NOTE)


In [ ]:
# 5. SOURCE_FILE_IDENTITY — **보고만 한다. 인증하지 않는다.**
#    등록 data.py 는 file id 로만 고른다. 이름이 같은 파일을 찾은 것은 증거가
#    아니다 — 그 치환을 막는 것이 이 PREP 의 존재 이유다.
#    인증·다운로드는 이 셀에 없다. 노트북이 먼저 adapter 를 만들면 terminal
#    guard 가 보지도 못한 채 credential 이 발급되기 때문이다.
print('registered file   :', P3.REGISTERED_SOURCE_NAME, '::',
      P3.REGISTERED_SOURCE_FUNCTION)
print('registered file id:', P3.REGISTERED_SOURCE_FILE_ID)
print('registered folder :', P3.REGISTERED_SOURCE_FOLDER_ID)
print('registered bytes  :', P3.REGISTERED_SOURCE_BYTES)
print('registered sha256 :', P3.REGISTERED_SOURCE_SHA256)
print('ASSETS row        :', P3.REGISTERED_SOURCE_ASSET_ROW)
print()
print('검증 순서: file id 직접 조회 → provider inventory(size/checksum/parents)')
print('          → 바이트 읽기 → 읽은 바이트의 SHA-256 → 그 다음에야 import')
print()
print('gate 순서:')
for index, gate in enumerate(P3.P3_GATE_ORDER, 1):
    print(f'  {index}. {gate}')
print()
print('중단 사유(등록 파일이 아니면 여기서 멈춘다):')
for stop in P3.HARNESS_STOPS:
    print('  -', stop)


In [ ]:
# 6. SOURCE_ORACLE_DIFFERENTIAL — 실제 실행 경로. 두 장벽이 모두 열려야 온다.
#    지금 상태로는 terminal guard 에서 거부되며, 그것이 커밋된 의도다.
#    adapter_source=None 이다: 인증과 adapter 생성은 guard **아래**
#    run_p3() 안에서만 일어난다. 노트북은 credential 을 만지지 않는다.
#
#    oracle 은 등록 build_record 자체다. 주입되는 것:
#      - wfdb reader stub (합성 ramp 신호, 실제 ECG 아님)
#      - detect_r stub    (fixture 의 peak, 실제 검출기 미실행)
#      - rr/feature stub  (행이 어느 peak 것인지 스스로 밝히는 행)
RESULT = None
OUT_DIR = '/content/drive/MyDrive/MedKOS/ecg-model/runs'
TIMESTAMP = ''      # 실행 시각(YYYYmmddTHHMMSS)을 승인 실행에서 채운다

if P3.execution_is_approved(APPROVAL) and OPEN_REGISTERED_DATA:
    RESULT = P3.run_p3(
        OUT_DIR, timestamp=TIMESTAMP,
        adapter_source=None,          # 인증은 guard 아래에서만
        approval=APPROVAL,
        open_registered_data=OPEN_REGISTERED_DATA,
        file_id=P3.REGISTERED_SOURCE_FILE_ID)
else:
    print('실행하지 않음 — 두 장벽 중 하나 이상이 닫혀 있다.')
    print('  OPEN_REGISTERED_DATA        :', OPEN_REGISTERED_DATA)
    print('  approval token 유효         :', P3.execution_is_approved(APPROVAL))
    print('  EXECUTION_APPROVAL_RECORD   :',
          P3.EXECUTION_APPROVAL_RECORD['granted'])
    print()
    print('승인 없이 여기까지 온 호출은 credential 0회 · Drive API 0회 ·')
    print('등록 바이트 0회 · 출력 디렉터리 생성 0회다.')


In [ ]:
# 7. RESULT_GATE — 판정과 후보 구조 검사. **등록은 하지 않는다.**
#    모든 fixture 가 일치할 때만 후보가 생기고, 그 후보를 현재 main 의
#    Q5E.verify_source_match_equivalence() 에 넣어 구조적으로 통과하는지만 본다.
#    SOURCE_MATCH_ORACLE_RECORD 에는 쓰지 않는다.
if RESULT is None:
    print('결과 없음 — 위 셀이 실행되지 않았다.')
    print('현재 SOURCE_MATCH_ORACLE_RECORD :', Q5E.SOURCE_MATCH_ORACLE_RECORD)
    print('현재 M4.0 sub-gate              :',
          Q5E.verify_source_match_equivalence()['reason'])
else:
    decision = RESULT['decision']
    print('status            :', decision['status'])
    print('harness stop      :', decision['harness_stop'])
    print('first failure     :', decision['first_failure'])
    print('fixtures passed   :', decision['fixtures_passed'], '/',
          decision['fixtures_total'])
    print()
    print('| fixture | source digest | adapter digest | equal |')
    print('|---|---|---|---|')
    for entry in (RESULT['differential'] or {}).get('fixtures', ()):
        print(f"| {entry['name']} | {entry['source_result_sha256']} "
              f"| {entry['adapter_result_sha256']} | {entry['equal']} |")
    print()
    gate = RESULT['candidate_gate']
    print('등록 gate 통과(구조) :', gate['ok'])
    for problem in gate['problems']:
        print('   -', problem)
    print('등록 상수에 기록함   :', gate['registered_constant_written'])
    if not decision['equivalence_claimed']:
        print()
        print('불일치가 하나라도 있으면: verdict 는')
        print('SOURCE_MATCH_EQUIVALENCE_REQUIRED 로 남고, real-record count 를')
        print('열지 않으며, 어댑터를 자동 수정하지 않는다. 수정은 별도 PR 이다.')


In [ ]:
# 8. BUNDLE_REPORT — 외부 동결용 보고. 이 셀의 **저장된 출력** 이
#    manifest SHA-256 의 외부 anchor 다(번들 안에는 자기 digest 를 적지 않는다).
#    후보 record 의 prep_bundle_sha256 은 이 번들의 payload fold 이므로,
#    후보 자체도 번들 안에 넣지 않고 여기서 보고한다.
if RESULT is None:
    print('보고할 번들이 없다 — 실행 승인 전이다.')
    print('실행되면 이 셀이 다음을 전부 찍는다:')
    for line in ('source file id 와 SHA-256 검증',
                 'adapter fingerprint',
                 'oracle harness SHA-256',
                 'fixture 별 source/adapter digest 와 equal',
                 'required fixture coverage',
                 'first failure',
                 'final verdict',
                 'prep payload fold 전체 64-hex',
                 'manifest SHA-256 전체 64-hex',
                 'Drive run folder 와 folder ID',
                 'SOURCE_MATCH_ORACLE_RECORD registration candidate',
                 '다음 단계가 registration 인지 adapter 수정인지'):
        print('  -', line)
else:
    inventory = RESULT['source_inventory']
    bundle = RESULT['bundle']
    harness = RESULT['harness']
    decision = RESULT['decision']
    print('== source identity ==')
    print('file id                :', inventory['requested_file_id'])
    print('provider sha256        :', inventory.get('provider_sha256'))
    print('읽은 바이트 sha256     :', inventory['observed_sha256'])
    print('등록 digest 와 일치    :', inventory['digest_matches_registered'])
    print()
    print('== identities ==')
    print('adapter fingerprint    :',
          Q5E.source_match_adapter_fingerprint())
    print('oracle harness SHA-256 :', harness['oracle_harness_sha256'])
    print()
    print('== fixtures ==')
    for entry in (RESULT['differential'] or {}).get('fixtures', ()):
        print(f"  {entry['name']}")
        print(f"    source : {entry['source_result_sha256']}")
        print(f"    adapter: {entry['adapter_result_sha256']}")
        print(f"    equal  : {entry['equal']}")
    print('required fixture coverage:',
          sorted(e['name'] for e in
                 (RESULT['differential'] or {}).get('fixtures', ())) ==
          sorted(Q5E.SOURCE_MATCH_REQUIRED_FIXTURES))
    print('first failure           :', decision['first_failure'])
    print('final verdict           :', decision['status'])
    print()
    print('== bundle ==')
    print('run folder             :', bundle['directory'])
    print('folder ID              : (Drive UI 에서 확인해 ASSETS 에 기록)')
    print('prep payload fold      :', bundle['prep_payload_sha256'])
    print('manifest SHA-256       :',
          bundle['manifest_sha256_freeze_externally'])
    print()
    print('== registration candidate (등록 아님) ==')
    print(json.dumps(RESULT['candidate'], indent=1, sort_keys=True))
    print('SOURCE_MATCH_ORACLE_RECORD 현재값 :',
          Q5E.SOURCE_MATCH_ORACLE_RECORD)
    print()
    print('다음 단계:', decision['next_step'])


## 실행 후 절차

1. 이 노트북을 **출력을 담아** `notebooks/executed/` 에 저장한다. 셀 8 의
   저장된 출력이 manifest SHA-256 의 외부 anchor 이고, 후보 record 가 남는
   유일한 곳이다.
2. Drive run folder 와 folder ID 를 `research/ASSETS.md` 에 등록한다.
3. Codex 가 번들을 읽기 전용으로 다시 가져와 결과를 인수한다.
4. 인수된 뒤에야 **별도 registration PR** 이
   `SOURCE_MATCH_ORACLE_RECORD` 를 채운다. 이 노트북은 채우지 않는다.

불일치가 하나라도 있었다면 3~4 대신: 불일치 trace 와 두 digest 를 보존한 채
Codex 재검토로 넘기고, 어댑터 정정은 별도 PR 에서 하고 전 fixture 를 처음부터
다시 돌린다. **PASS 를 주장하지 않는다.**
